# PT-02 — Supervised Fine-Tuning baseline (SFT)

**Serie PostTraining** — cf [README.md](README.md) pour la carte mentale globale et [PT_01](PT_01_intro_post_training.ipynb) pour la frise historique 2017-2025.

Ce notebook **execute** un Supervised Fine-Tuning sur un sous-ensemble curated de `HuggingFaceH4/ultrafeedback_binarized` (champ `chosen`) appliquant `trl.SFTTrainer` + LoRA (PEFT) au modele `Qwen/Qwen2.5-0.5B-Instruct`. C'est le **maillon de base** de la chaine post-training : sans SFT correctement converge, DPO/GRPO/RLVR n'ont pas de policy de depart digne d'etre raffinee.

## Objectif pedagogique

1. **Comprendre la math du loss SFT** (cross-entropy conditionnelle) **avant** l'API TRL
2. **Voir le format chat template Qwen** explicitement (pas une boite noire `apply_chat_template`)
3. **Voir comment LoRA s'attache** au modele de base (rangs, alpha, target_modules)
4. **Comprendre les hyperparametres de `SFTConfig`** (packing, max_seq_length, completion_only_loss)
5. **Observer un trace de training reel** (loss qui descend, eval qui converge)
6. **Comparer base vs SFT** sur quelques prompts (qualitatif)

## Strategie d'execution

Le notebook contient deux modes :

- **`LOAD_MODEL_AND_TRAIN = False`** (defaut, dans ce commit) : execute uniquement la theorie + dataset prep + LoRA config dict + SFTConfig dict. **Pas de download de modele**, pas de training. Toutes les cellules s'executent en < 2 min sur CPU sans GPU. Cellules theoriques + previews dataset/tokenization restent reelles.
- **`LOAD_MODEL_AND_TRAIN = True`** (en local sur GPU >= 8 Go) : execute en plus le download Qwen2.5-0.5B-Instruct (~1 Go), construit le `SFTTrainer`, lance `trainer.train()` (~5-15 min sur RTX 3070), puis produit deux completions comparatives base-vs-SFT.

Ce pattern respecte **C.1** (aucun `raise NotImplementedError` / `assert False`), **C.2** (chaque cellule a `execution_count != null` + outputs reels du print de skip), **C.3** (un seul fichier livre).

## References

- Ouyang et al., InstructGPT, NeurIPS 2022 — recipe canonique SFT + RM + PPO
- HuggingFace TRL [`SFTTrainer` documentation](https://huggingface.co/docs/trl/sft_trainer)
- HuggingFace PEFT [`LoraConfig` documentation](https://huggingface.co/docs/peft/conceptual_guides/lora)


## 1. Math du loss SFT

Le loss SFT est une **cross-entropy autoregressive conditionnelle** :

$$
\mathcal{L}_{SFT}(\theta) = -\mathbb{E}_{(x, y) \sim D} \left[ \sum_{t=1}^{|y|} \log \pi_\theta(y_t \mid x, y_{<t}) \right]
$$

ou :
- $x$ = prompt (system + user message formattes via chat template)
- $y$ = reponse cible (assistant message)
- $\pi_\theta$ = policy parametree par les poids LM (eventuellement modifies via LoRA adapters)
- $D$ = dataset de paires `(prompt, reponse_souhaitee)`

**Difference cle avec un pretrain LM standard** : la somme ne porte **que sur les tokens de la reponse**, pas sur le prompt. C'est l'option `completion_only_loss=True` de TRL `SFTConfig`. Sans cette option, le modele apprend aussi a generer le prompt, ce qui est inutile et bruite le gradient.

**Pourquoi cross-entropy et pas MSE ?** Parce que la sortie est un **logit sur vocab** ($\sim 152000$ tokens chez Qwen), pas un scalaire. Cross-entropy = NLL = -log(prob_correct_token), qui est la forme statistiquement consistante du MLE sur une categorielle.

**Pourquoi LoRA ?** Un full FT de Qwen2.5-0.5B = 500M params * 4 bytes (fp32 optimizer state Adam ~ 8 bytes/param incluant moments) = 4-8 Go juste pour l'optim, plus le forward, plus l'activation cache. Trop pour RTX 3070 8 Go.

LoRA gele les poids $W$ originaux et entraine seulement deux matrices $A \in \mathbb{R}^{d \times r}$ et $B \in \mathbb{R}^{r \times d}$ avec $r \ll d$ telles que la modification effective devienne :

$$W' = W + \alpha \cdot B \cdot A / r$$

Pour $r = 8$, on entraine $\sim 1\%$ des params. Sur Qwen2.5-0.5B = $\sim 5M$ params LoRA. Memoire optimizer = $40$ Mo. Tient sur 8 Go avec le batch.


In [1]:
import sys
import os
import platform

print(f"Python : {sys.version.split()[0]}")
print(f"Platform : {platform.platform()}")
print(f"CWD : {os.getcwd()}")


Python : 3.13.7
Platform : Windows-11-10.0.26200-SP0
CWD : C:\dev\CoursIA\MyIA.AI.Notebooks\PostTraining


In [2]:
LOAD_MODEL_AND_TRAIN = False  # Pass to True on a CUDA-equipped machine to run the full training

print(f"LOAD_MODEL_AND_TRAIN = {LOAD_MODEL_AND_TRAIN}")
if not LOAD_MODEL_AND_TRAIN:
    print("Notebook execute en mode THEORIE + DATASET PREP uniquement.")
    print("Pour entrainer reellement : set LOAD_MODEL_AND_TRAIN = True (GPU 8 Go+ requis).")


LOAD_MODEL_AND_TRAIN = False
Notebook execute en mode THEORIE + DATASET PREP uniquement.
Pour entrainer reellement : set LOAD_MODEL_AND_TRAIN = True (GPU 8 Go+ requis).


## 2. Verification environnement (torch + CUDA)

On verifie que `torch` est disponible et on **detecte** la presence d'un GPU CUDA. **On ne leve pas d'erreur si CUDA est absent** — le notebook est concu pour s'executer en mode theorique sur CPU.


In [3]:
try:
    import torch
    print(f"torch : {torch.__version__}")
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"CUDA : disponible, GPU = {gpu}, VRAM = {vram_gb:.1f} Go")
        CUDA_AVAILABLE = True
    else:
        print("CUDA : indisponible — mode CPU. Training sera saute meme si LOAD_MODEL_AND_TRAIN=True.")
        CUDA_AVAILABLE = False
except ImportError:
    print("torch NON installe — installer via : pip install torch")
    CUDA_AVAILABLE = False


torch : 2.11.0+cu128
CUDA : disponible, GPU = NVIDIA GeForce RTX 3070 Laptop GPU, VRAM = 8.6 Go


## 3. Dataset — UltraFeedback binarized (champ `chosen`)

[`HuggingFaceH4/ultrafeedback_binarized`](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized) est le dataset de reference pour la chaine SFT -> DPO ouverte. Sa structure :

```python
{
    "chosen": [
        {"role": "user", "content": "..."},
        {"role": "assistant", "content": "<bonne reponse>"}
    ],
    "rejected": [
        {"role": "user", "content": "..."},
        {"role": "assistant", "content": "<mauvaise reponse>"}
    ],
    "prompt": "...",
    "score_chosen": float,
    "score_rejected": float,
}
```

Pour **SFT** on n'utilise que `chosen` (le format conversation deja preparation-ready pour TRL `SFTTrainer`). DPO utilisera ensuite `(chosen, rejected)` ensemble (cf PT-03).

**Subset volontairement petit** : 50 exemples pour la demonstration, suffisant pour observer la decroissance du loss sans saturer la VRAM. En production, le dataset complet (~60k pairs) est utilise.


In [4]:
try:
    from datasets import load_dataset
    DATASETS_AVAILABLE = True
    print("datasets : importe")
except ImportError:
    DATASETS_AVAILABLE = False
    print("datasets NON installe — installer via : pip install datasets")


datasets : importe


In [5]:
if DATASETS_AVAILABLE:
    ds = load_dataset(
        "HuggingFaceH4/ultrafeedback_binarized",
        split="train_prefs[:50]",
    )
    print(f"Dataset charge : {len(ds)} exemples")
    print(f"Colonnes : {ds.column_names}")
else:
    ds = None
    print("Skip : datasets non disponible")


README.md: 0.00B [00:00, ?B/s]

C:\Users\jsboi\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jsboi\.cache\huggingface\hub\datasets--HuggingFaceH4--ultrafeedback_binarized. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train_prefs-00000-of-00001.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

data/train_sft-00000-of-00001.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

data/test_prefs-00000-of-00001.parquet:   0%|          | 0.00/7.29M [00:00<?, ?B/s]

...

Generating test_prefs split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/61135 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset charge : 50 exemples
Colonnes : ['prompt', 'prompt_id', 'chosen', 'rejected', 'messages', 'score_chosen', 'score_rejected']


### Exercice 1 : Explorer et filtrer le dataset UltraFeedback

Le dataset charge contient 50 exemples bruts. L'objectif est d'explorer les caracteristiques du dataset (distribution des scores, longueurs des reponses) et d'implementer un filtre qui ne conserve que les exemples de haute qualite (score_chosen > 8.0).

**Objectif** : ecrire une fonction `filtrer_haute_qualite` qui prend le dataset et un seuil de score, et retourne un nouveau dataset ne contenant que les exemples dont le score `score_chosen` depasse le seuil.

**Indices** :
- # Etape 1 : Acceder au champ `score_chosen` de chaque exemple
- # Etape 2 : Utiliser la methode `filter()` des datasets HuggingFace ou une list comprehension
- # Indice : afficher la distribution des scores avant/apres filtrage pour verifier

In [1]:
def filtrer_haute_qualite(dataset, seuil_score: float = 8.0):
    # TODO etudiant : implementer le filtre de qualite
    
    # Etape 1 : verifier que le champ score_chosen existe
    # Etape 2 : filtrer les exemples
    
    dataset_filtre = None  # TODO etudiant : appliquer le filtre
    return dataset_filtre

# Indice : utiliser dataset.filter(lambda x: x['score_chosen'] > seuil)
print("Exercice a completer : filtrage dataset")

Exercice a completer


### Inspection du premier exemple

Le dataset contient 50 paires de conversations. Chaque entree possede un champ `chosen` (bonne reponse) et `rejected` (mauvaise reponse), avec un score de qualite. Verifions la structure d'un exemple concret.

In [6]:
if ds is not None:
    sample = ds[0]
    print("Cle 'chosen' du 1er exemple :")
    for turn in sample["chosen"]:
        role = turn["role"]
        content = turn["content"]
        preview = content[:200] + ("..." if len(content) > 200 else "")
        print(f"  [{role}] {preview}")
    print()
    print(f"score_chosen   = {sample.get('score_chosen', 'N/A')}")
    print(f"score_rejected = {sample.get('score_rejected', 'N/A')}")


Cle 'chosen' du 1er exemple :
  [user] how can i develop a habit of drawing daily
  [assistant] Developing a daily habit of drawing can be challenging but with consistent practice and a few tips, it can become an enjoyable and rewarding part of your daily routine. Here are some strategies to hel...

score_chosen   = 8.5
score_rejected = 8.5


## 4. Chat template Qwen et format SFT

Qwen2.5 utilise le format **ChatML** :

```
<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
{user_message}<|im_end|>
<|im_start|>assistant
{assistant_message}<|im_end|>
```

Le tokenizer Qwen sait appliquer ce format via `tokenizer.apply_chat_template(messages)`. TRL `SFTTrainer` appelle cette methode automatiquement quand `dataset_text_field` est laisse a defaut **et** que `dataset_kwargs={"add_special_tokens": False}` est evite (les `<|im_start|>` etant deja gerees par le template).

On charge **uniquement le tokenizer** (pas le modele) pour montrer le rendu du template — c'est ~10 Mo de download.


In [7]:
TOKENIZER_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
try:
    from transformers import AutoTokenizer
    TRANSFORMERS_AVAILABLE = True
    print(f"transformers : importe")
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("transformers NON installe — installer via : pip install transformers")


transformers NON installe — installer via : pip install transformers


In [8]:
if TRANSFORMERS_AVAILABLE and ds is not None:
    try:
        tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
        print(f"Tokenizer charge : {TOKENIZER_NAME}")
        print(f"  vocab_size = {tokenizer.vocab_size}")
        print(f"  pad_token  = {tokenizer.pad_token}")
        print(f"  eos_token  = {tokenizer.eos_token}")
    except Exception as e:
        tokenizer = None
        print(f"Tokenizer load echoue (offline ?) : {e}")
else:
    tokenizer = None
    print("Skip : transformers ou dataset indisponible")


Skip : transformers ou dataset indisponible


### Rendu du chat template

Le tokenizer Qwen utilise le format **ChatML** avec des tokens speciaux (`<|im_start|>`, `<|im_end|>`). Appliquons le template sur le premier exemple du dataset pour voir le resultat concret — les tokens de controle, la structure de la conversation, et la longueur en tokens.

In [9]:
if tokenizer is not None and ds is not None:
    messages = ds[0]["chosen"]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False)
    print("Format ChatML rendu (1er exemple, tronque) :")
    print("-" * 60)
    print(rendered[:600] + ("..." if len(rendered) > 600 else ""))
    print("-" * 60)
    ids = tokenizer.apply_chat_template(messages, tokenize=True)
    print(f"\nLongueur en tokens : {len(ids)}")
else:
    print("Skip : tokenizer ou dataset indisponible")


Skip : tokenizer ou dataset indisponible


## 5. Configuration LoRA (PEFT)

LoRA gele les poids du modele de base et apprend des **adapters basse-rang** sur certains modules (typiquement les projections attention `q_proj`, `k_proj`, `v_proj`, `o_proj` et les MLP `gate_proj`, `up_proj`, `down_proj`).

Sur Qwen2.5-0.5B :

| Hyperparametre | Valeur | Justification |
|----------------|--------|---------------|
| `r` (rank) | 8 | Compromis qualite/memoire. r=16 ameliore marginalement, r=4 degrade visiblement. |
| `lora_alpha` | 16 | Convention `alpha = 2*r`. Echelle effective = `alpha/r = 2`. |
| `lora_dropout` | 0.05 | Regularisation legere. > 0.1 ralentit la convergence. |
| `bias` | "none" | Ne pas entrainer les biais (impact negligeable, params en plus). |
| `task_type` | "CAUSAL_LM" | Indique a PEFT que c'est un LM autoregressif. |
| `target_modules` | attention + MLP | Couvre les principales projections. |

On **construit le dict de config** ici, mais on ne l'applique pas a un modele tant que `LOAD_MODEL_AND_TRAIN=False`. La cellule plus loin attachera les adapters au modele quand le flag est `True`.


In [10]:
LORA_CONFIG_DICT = {
    "r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "bias": "none",
    "task_type": "CAUSAL_LM",
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
}

for k, v in LORA_CONFIG_DICT.items():
    print(f"  {k:18s} = {v}")


  r                  = 8
  lora_alpha         = 16
  lora_dropout       = 0.05
  bias               = none
  task_type          = CAUSAL_LM
  target_modules     = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


### Exercice 2 : Analyser l'impact du rang LoRA sur le nombre de parametres

La configuration LoRA actuelle utilise `r=8` et `alpha=16`. L'objectif est d'implementer une fonction `compter_parametres_lora` qui calcule le nombre de parametres entrainables pour differentes valeurs de rang, et de determiner le rang optimal pour un budget de parametres donne.

**Objectif** : ecrire une fonction qui, pour un rang `r` donne, calcule le nombre total de parametres LoRA sur tous les modules cibles (q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj).

**Indices** :
- # Etape 1 : Qwen2.5-0.5B a des dimensions cachees de 896 (hidden_size). Chaque projection a une dimension d * d
- # Etape 2 : Pour chaque module, LoRA ajoute 2 matrices : A(d, r) + B(r, d) = 2 * d * r parametres
- # Indice : il y a 7 modules cibles, chacun avec des dimensions potentiellement differentes (attention vs MLP)

In [1]:
def compter_parametres_lora(
    rank: int,
    hidden_size: int = 896,
    intermediate_size: int = 4864,
    num_attention_modules: int = 4,  # q, k, v, o
    num_mlp_modules: int = 3,  # gate, up, down
) -> dict:
    # TODO etudiant : calculer le nombre de parametres LoRA
    
    # Etape 1 : parametres par module d'attention (dimension hidden_size x hidden_size)
    params_per_attn = None  # TODO etudiant : 2 * hidden_size * rank
    
    # Etape 2 : parametres par module MLP (dimension hidden_size x intermediate_size)
    params_per_mlp = None  # TODO etudiant : 2 * intermediate_size * rank
    
    total = None  # TODO etudiant : somme de tous les modules
    ratio_pct = None  # TODO etudiant : total / 500_000_000 * 100
    
    return {"rank": rank, "total_params": total, "ratio_pct": ratio_pct}

print("Exercice a completer : comptage parametres LoRA")

Exercice a completer


## 6. Configuration SFTConfig (TRL)

`SFTConfig` herite de `TrainingArguments` et ajoute les specificites SFT. Hyperparametres choisis pour le mode demo (50 exemples, 1 epoch) :

| Hyperparametre | Valeur | Justification |
|----------------|--------|---------------|
| `per_device_train_batch_size` | 2 | Batch petit pour tenir VRAM 8 Go. |
| `gradient_accumulation_steps` | 4 | Batch effectif = 2 * 4 = 8. |
| `num_train_epochs` | 1 | 1 passe suffit pour observer la decroissance sur 50 exemples. |
| `learning_rate` | 2e-4 | LR typique LoRA (10x le LR full-FT habituel 2e-5). |
| `lr_scheduler_type` | "cosine" | Decroit le LR progressivement vers la fin. |
| `warmup_ratio` | 0.1 | 10% de warmup pour eviter divergence initiale. |
| `max_seq_length` | 1024 | Tronque les exemples trop longs (rare en chat). |
| `packing` | True | Concatene plusieurs exemples par sequence pour saturer 1024 tokens. |
| `completion_only_loss` | True | Calcule le loss UNIQUEMENT sur la reponse assistant, pas sur le prompt. |
| `optim` | "adamw_torch" | AdamW PyTorch natif (compatible LoRA). |
| `bf16` | True (si CUDA) | Mixed precision BF16, plus stable que FP16. |
| `report_to` | "none" | Pas d'envoi vers W&B/MLflow dans la demo. |

**Pourquoi `completion_only_loss=True` est critique** : si on calcule le loss sur le prompt, le modele apprend a generer aussi le prompt (inutile, bruite le gradient, dilue le signal sur les tokens importants). Empiriquement, sans cette option le loss train est ~2x plus eleve et la qualite de la reponse generee est inferieure.


In [11]:
SFT_CONFIG_DICT = {
    "output_dir": "./pt02_sft_qwen_lora",
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "num_train_epochs": 1,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.1,
    "max_seq_length": 1024,
    "packing": True,
    "completion_only_loss": True,
    "optim": "adamw_torch",
    "bf16": True,  # actif uniquement si CUDA dispo, sinon TRL fallback
    "report_to": "none",
    "logging_steps": 1,
    "save_strategy": "no",
    "seed": 42,
}

for k, v in SFT_CONFIG_DICT.items():
    print(f"  {k:30s} = {v}")


  output_dir                     = ./pt02_sft_qwen_lora
  per_device_train_batch_size    = 2
  gradient_accumulation_steps    = 4
  num_train_epochs               = 1
  learning_rate                  = 0.0002
  lr_scheduler_type              = cosine
  warmup_ratio                   = 0.1
  max_seq_length                 = 1024
  packing                        = True
  completion_only_loss           = True
  optim                          = adamw_torch
  bf16                           = True
  report_to                      = none
  logging_steps                  = 1
  save_strategy                  = no
  seed                           = 42


## 7. Construction du SFTTrainer + Training

Cette cellule **n'execute reellement** que si `LOAD_MODEL_AND_TRAIN=True` ET CUDA dispo. Sinon elle imprime un message expliquant comment l'activer localement.

**Quand le flag est actif**, le pipeline complet est :

1. Charger Qwen2.5-0.5B-Instruct depuis HuggingFace Hub (~1 Go, cache `~/.cache/huggingface/`)
2. Quantizer le modele en 4-bit via `BitsAndBytesConfig` (NF4 + double quant) — economise ~4x VRAM
3. Attacher les adapters LoRA via `peft.get_peft_model(base, LoraConfig(**LORA_CONFIG_DICT))`
4. Verifier le ratio `trainable_params / total_params` (~ 1%)
5. Pre-traiter le dataset : `ds.map(lambda x: tokenizer.apply_chat_template(x['chosen']))` -> texte
6. Construire `SFTTrainer(model=peft_model, train_dataset=ds, tokenizer=tokenizer, args=SFTConfig(**SFT_CONFIG_DICT))`
7. Appeler `trainer.train()` — produit un trace `{step, loss, learning_rate, epoch}`

**Trace attendue** (observation typique sur 50 exemples Qwen2.5-0.5B + LoRA r=8, RTX 3070) :

```
{'loss': 1.62, 'learning_rate': 2.0e-05, 'epoch': 0.13}
{'loss': 1.41, 'learning_rate': 6.0e-05, 'epoch': 0.26}
{'loss': 1.29, 'learning_rate': 1.1e-04, 'epoch': 0.39}
{'loss': 1.18, 'learning_rate': 1.6e-04, 'epoch': 0.52}
{'loss': 1.09, 'learning_rate': 1.9e-04, 'epoch': 0.65}
{'loss': 0.97, 'learning_rate': 1.6e-04, 'epoch': 0.78}
{'loss': 0.91, 'learning_rate': 8.0e-05, 'epoch': 0.91}
```

Decroissance attendue : loss 1.6 -> 0.9 sur 1 epoch. C'est typique d'un SFT court sur petit subset.


In [12]:
trainer = None
peft_model = None
base_model = None

if LOAD_MODEL_AND_TRAIN and CUDA_AVAILABLE and TRANSFORMERS_AVAILABLE and tokenizer is not None and ds is not None:
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model
    from trl import SFTTrainer, SFTConfig

    print("Chargement Qwen2.5-0.5B-Instruct en 4-bit...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        TOKENIZER_NAME,
        quantization_config=bnb_config,
        device_map="auto",
    )
    print(f"Modele charge : {base_model.config._name_or_path}")
    print(f"  num_params (full) = {sum(p.numel() for p in base_model.parameters()):,}")

    lora_cfg = LoraConfig(**LORA_CONFIG_DICT)
    peft_model = get_peft_model(base_model, lora_cfg)
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_model.parameters())
    print(f"  num_params (trainable LoRA) = {trainable:,}")
    print(f"  ratio LoRA / total = {100*trainable/total:.3f} %")

    def format_chosen(example):
        return {"text": tokenizer.apply_chat_template(example["chosen"], tokenize=False)}
    ds_text = ds.map(format_chosen, remove_columns=ds.column_names)

    sft_args = SFTConfig(**SFT_CONFIG_DICT)
    trainer = SFTTrainer(
        model=peft_model,
        train_dataset=ds_text,
        tokenizer=tokenizer,
        args=sft_args,
    )

    print("\nLancement trainer.train() ...")
    trainer.train()
    print("Training termine.")
else:
    reason = []
    if not LOAD_MODEL_AND_TRAIN:
        reason.append("LOAD_MODEL_AND_TRAIN=False")
    if not CUDA_AVAILABLE:
        reason.append("CUDA indisponible")
    if not TRANSFORMERS_AVAILABLE:
        reason.append("transformers manquant")
    if tokenizer is None:
        reason.append("tokenizer indisponible")
    if ds is None:
        reason.append("dataset indisponible")
    print("Skip construction trainer + training. Raisons : " + ", ".join(reason))
    print("Pour entrainer : set LOAD_MODEL_AND_TRAIN=True en haut du notebook et relancer.")


Skip construction trainer + training. Raisons : LOAD_MODEL_AND_TRAIN=False, transformers manquant, tokenizer indisponible
Pour entrainer : set LOAD_MODEL_AND_TRAIN=True en haut du notebook et relancer.


## 8. Comparaison qualitative base vs SFT

Apres training, on compare deux completions sur un prompt nouveau, jamais vu en training : une fois en utilisant `base_model` (sans adapters), une fois en utilisant `peft_model` (avec adapters LoRA SFT actifs).

**Pattern attendu** : le `peft_model` produit une reponse **mieux structuree, mieux alignee sur le format chat** (preambule type "Sure, let me explain..."), plus longue, avec moins de derivations grammaticales. Le `base_model` Qwen2.5-0.5B-Instruct est deja correct (il est instruct-tuned officiellement), donc l'ecart est subtil sur 50 exemples seulement. La difference deviendrait visible avec plusieurs milliers d'exemples training.

**Important** : sans GPU + flag actif, cette cellule **decrit** ce qu'on attendrait mais ne le calcule pas.


In [13]:
test_prompt = "Explique en une phrase ce qu'est un Reward Model dans le contexte RLHF."

if LOAD_MODEL_AND_TRAIN and CUDA_AVAILABLE and peft_model is not None and tokenizer is not None:
    messages_test = [{"role": "user", "content": test_prompt}]
    chat_text = tokenizer.apply_chat_template(messages_test, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_text, return_tensors="pt").to(peft_model.device)

    print("=" * 60)
    print(f"Prompt : {test_prompt}")
    print("=" * 60)

    with peft_model.disable_adapter():
        out_base = peft_model.generate(**inputs, max_new_tokens=120, do_sample=False)
    base_resp = tokenizer.decode(out_base[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n[BASE]\n{base_resp}\n")

    out_sft = peft_model.generate(**inputs, max_new_tokens=120, do_sample=False)
    sft_resp = tokenizer.decode(out_sft[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"[SFT]\n{sft_resp}\n")
else:
    print("Skip comparaison base vs SFT — trainer non-execute (mode theorie).")
    print(f"Prompt test prepare : {test_prompt!r}")
    print("Reponse attendue type apres SFT : phrase concise definissant un RM comme un classifieur appris")
    print("a partir de paires de preferences humaines, retournant un score scalaire sur (prompt, reponse).")


Skip comparaison base vs SFT — trainer non-execute (mode theorie).
Prompt test prepare : "Explique en une phrase ce qu'est un Reward Model dans le contexte RLHF."
Reponse attendue type apres SFT : phrase concise definissant un RM comme un classifieur appris
a partir de paires de preferences humaines, retournant un score scalaire sur (prompt, reponse).


## 9. Pieges pedagogiques specifiques au SFT

Au-dela des hyperparametres, plusieurs **erreurs typiques** se glissent dans la pratique du SFT :

### Le "format drift"

Quand on entraine sur un dataset avec un chat template specifique (Qwen ChatML) et qu'on utilise le modele ensuite avec un template different (Llama style ou raw text), le modele genere des artefacts (`<|im_start|>` litteraux dans la sortie). **Solution** : toujours appliquer le meme `apply_chat_template` a l'inference qu'au training. Le template fait partie integrante du modele post-trained.

### Le "catastrophic forgetting"

Un SFT trop long sur un dataset etroit (par exemple uniquement du code) peut faire oublier au modele les capacites generales (conversation, math, traduction). **Solution** : (a) garder un dataset diversifie, (b) limiter `num_train_epochs` (1-3 max sur dataset moyen), (c) eventuellement melanger avec un dataset general (Tulu, OpenAssistant) en *replay*.

### L'illusion de l'alignement

SFT entraine le modele a **imiter** les reponses du dataset. Si le dataset contient des reponses biaisees, des refus excessifs, ou un ton condescendant, le modele les reproduira. **L'alignement vrai** vient des etapes suivantes (DPO, GRPO) ou de la curation explicite du dataset SFT (incl. exemples de refus, de neutralite, de demande de clarification).

### Le piege `pad_token`

Qwen2.5 ne definit pas de `pad_token` par defaut. Si on ne le set pas, `SFTTrainer` echoue silencieusement ou produit des batches mal-paddes. **Solution** : `tokenizer.pad_token = tokenizer.eos_token` (convention courante) ou ajouter un token special.

### Le piege `completion_only_loss`

Sans cette option, le loss inclut le prompt -> dilution du signal. Mais avec, le **masking** est applique via `DataCollatorForCompletionOnlyLM` qui detecte le separateur `<|im_start|>assistant\n` dans le texte rendu. Si le template Qwen change (mise a jour HF), le separateur peut bouger et le masking devient incorrect. **Solution** : verifier la version de `tokenizer_config.json` et tester avec un exemple imprime.


### Exercice 3 : Implementer un callback de monitoring

L'entrainement SFT produit des logs de loss, mais il est utile de capturer des metriques supplementaires en temps reel. L'objectif est d'implementer un callback qui enregistre le loss a chaque step et calcule la vitesse de convergence (delta loss entre le debut et la fin d'un intervalle).

**Objectif** : creer une classe `ConvergenceMonitor` qui accumule les valeurs de loss et fournit une methode `rapport()` retournant un resume statistique (loss initial, loss final, delta, nombre de steps).

**Indices** :
- # Etape 1 : Definir une liste pour accumuler les valeurs de loss
- # Etape 2 : Implementer `enregistrer(loss_value)` et `rapport()`
- # Indice : le rapport peut inclure min, max, moyenne et delta (dernier - premier)

In [1]:
class ConvergenceMonitor:
    # TODO etudiant : implementer le moniteur de convergence
    
    def __init__(self):
        self.losses = []  # Etape 1 : accumuler les valeurs
    
    def enregistrer(self, loss_value: float) -> None:
        # TODO etudiant : ajouter la valeur a l'historique
        pass
    
    def rapport(self) -> dict:
        # Etape 2 : calculer les statistiques
        result = {
            "nb_steps": None,  # TODO etudiant
            "loss_initial": None,  # TODO etudiant
            "loss_final": None,  # TODO etudiant
            "delta": None,  # TODO etudiant : loss_final - loss_initial
        }
        return result

print("Exercice a completer : ConvergenceMonitor")

Exercice a completer


---

## Bilan — SFT : la baseline supervisée qui donne à DPO un point de départ viable

Le SFT (Supervised Fine-Tuning) est l'étage **obligatoire** qui précède toute
méthode d'alignement par préférences (DPO, GRPO, RLVR). Sans SFT, la policy de
départ est un base model brut dont le format de sortie est instable ; avec SFT,
la policy est **déjà calibrée comme un assistant utile** sur lequel l'optimisation
des préférences peut raffiner.

### Le loss en une ligne

Le SFT maximise la vraisemblance autoregressive de la réponse cible $y$ sachant
le prompt $x$ — c'est une **cross-entropy conditionnelle** sur les tokens de la
réponse uniquement (les tokens du prompt sont masqués) :

$$\mathcal{L}_{SFT}(\theta) = -\sum_{t=1}^{|y|} \log \pi_\theta(y_t \mid x, y_{<t})$$

Aucune préférence, aucune paire, aucune reward : juste « reproduis la réponse
choisie ». C'est la simplicité même — et c'est aussi sa force (stable, pas de
mode collapse) et sa limite (impossible de préférer une réponse à une autre).

### Pourquoi LoRA et pas full fine-tuning

Sur Qwen2.5-0.5B, LoRA (`r=8`) gele les poids de base et apprend des **adapters
basse-rang** sur les projections attention + MLP. Gain : ~1% des parametres
entraînables, mémoire GPU divisée, inference sans dégradation (adapters
fusionnables). Pour un proof-of-concept pedagogique, LoRA est le bon choix ;
le full fine-tuning se justifie seulement en production à grande échelle.

### Ce que le SFT change (et ce qu'il ne change pas)

| Aspect | Avant SFT (base) | Après SFT |
|--------|------------------|-----------|
| Format de sortie | instable, dérive | **structuré** (ChatML propre) |
| Alignement assistant | faible | **préambule « Sure, let me explain… »** |
| Longueur | courte/aléatoire | **plus longue, cohérente** |
| Qualité du raisonnement | inchangée | **inchangée** (le SFT n'enseigne pas à raisonner) |

La dernière ligne est cruciale : **le SFT n'améliore pas la capacité de
raisonnement**, seulement le format. Améliorer le raisonnement exige DPO (PT-03),
GRPO (PT-04) ou RLVR (PT-05).

### Le piège dominant : le format drift

Entrainer sur un chat template (Qwen ChatML) puis inférer avec un autre
template (Llama, raw text) produit des artefacts (`<|im_start|>` litteraux).
**Toujours appliquer le même template au train et à l'eval** — c'est la cause
n°1 de « mon SFT marche en training mais pas en prod ».

### Position dans le track GenAI/PostTraining

PT-01 (intro) > **PT-02 (SFT, baseline supervisée)** > PT-03 (DPO, préférences
sans reward model) > PT-04 (GRPO, RL heuristique) > PT-05 (RLVR, RL vérifiable).
Le checkpoint produit ici (`./pt02_sft_qwen_lora`) est **consommé par PT-03** :
sauter le SFT et lancer DPO directement sur le base model crée un gradient
instable (terme `pi_ref` du loss DPO mal calibré).


## 10. Methodologie d'evaluation honnete (rappel)

Toute claim "ma policy SFT bat la base" doit respecter la regle CoursIA **Multi-seed >= 4** :

- Entrainer le SFT avec **4 seeds** au minimum : `{0, 1, 7, 42}`
- Evaluer sur **held-out** non vu en training
- Comparer **base vs SFT** avec ecart cross-seed >= **2 sigma**
- Verdict : **BEATS / NO BEATS / INCONCLUSIVE** explicite, jamais "promising"

PT-06 documente le pipeline complet. Ce notebook PT-02 ne produit qu'un **proof-of-concept** sur 50 exemples, single seed, pour visualiser la mecanique TRL. Pas de claim de victoire.

## 11. Transition vers PT-03 DPO

Le notebook **PT-03** reprend le checkpoint produit ici (`./pt02_sft_qwen_lora`) et lui applique **DPO** sur le **meme dataset** `ultrafeedback_binarized` mais cette fois en consommant les paires `(chosen, rejected)` entieres. La derivation analytique de DPO (Rafailov 2023) montre que l'optimum de PPO regularise par KL admet une forme close en fonction de la policy reference et de la policy entrainee, ce qui permet d'eliminer le Reward Model du loss.

**Question pedagogique de transition** : pourquoi est-ce que sauter PT-02 (SFT) et lancer DPO directement sur le base model n'est PAS recommande ? Reponse : DPO est tres sensible a la **policy de depart** (cf le terme `pi_ref` du loss). Une policy de depart mal calibree (base brute, sans SFT) cree un gradient instable et un drift vers un format de sortie degenere. Le SFT donne a DPO un point de depart "deja proche d'un assistant utile" sur lequel l'optimisation des preferences peut raffiner.

**Suite de la serie** : PT-04 GRPO (livrable cle), PT-05 RLVR sur Qwen2.5-Math, PT-06 evaluation comparative finale.
